# Document Translation Pipeline

Section 1 runs without Ollama. Sections 2+ require `ollama serve` running with `translategemma` pulled.

In [35]:
import os, tempfile
from pathlib import Path

DOCX_PATH = "./15.06.2026 Outline Blueprint completo - JAG VK -FA-.docx"
SHORT_DOCX_PATH = "./docs/short-docs/29.01. Blueprint Cejil comenta VRA_short2.docx"

## 1. Glossary unit tests (no Ollama required)

### 1.1 Build a glossary and check prompt-section filtering
Only entries whose source term appears in the chunk should be injected.

In [2]:
from glossary import DomainGlossary, GlossaryEntry

g = DomainGlossary([
    GlossaryEntry(["PDDH"], "Human Rights Defenders (HRDs)",
                  target_alts=["HRDs"], kind="require"),
    GlossaryEntry(["Corte IDH"], "IACtHR",
                  target_alts=["Inter-American Court"], kind="require"),
    GlossaryEntry(["protectDefenders.eu"], "protectDefenders.eu", kind="verbatim"),
    GlossaryEntry(["blueprint"], "blueprint", kind="verbatim"),
])

# Chunk mentions PDDH and blueprint only — Corte IDH and protectDefenders.eu should be filtered out
chunk = "La PDDH presentó el blueprint ante la asamblea."
section = g.prompt_section(chunk)
print(section)

assert "PDDH" in section
assert "blueprint" in section
assert "Corte IDH" not in section
assert "protectDefenders.eu" not in section

Required terminology — use these translations exactly whenever the source term appears:
  PDDH → Human Rights Defenders (HRDs)
  blueprint → blueprint  (preserve exactly as written)



### 1.2 Violation detection
A violation fires when the source term is present but no accepted target form appears in the translation.

In [3]:
source = "La PDDH presentó un informe ante la Corte IDH. Visite protectDefenders.eu."

# Bad translation: PDDH mistranslated, Corte IDH dropped, URL altered
bad = "The women presented a report to the Court. See defenders.eu."
violations = g.violations(source, bad)
for v in violations:
    print(" -", v)
assert len(violations) == 3, f"expected 3 violations, got {len(violations)}"
print("OK — 3 violations detected")

# Good translation: hits target_alts for PDDH and Corte IDH, preserves URL exactly
good = "The HRDs presented a report to the Inter-American Court. Visit protectDefenders.eu."
violations_good = g.violations(source, good)
assert violations_good == [], f"expected 0 violations, got {violations_good}"
print("OK — 0 violations on correct translation")

 - 'PDDH' must translate to Human Rights Defenders (HRDs) (or: HRDs)
 - 'Corte IDH' must translate to IACtHR (or: Inter-American Court)
 - 'protectDefenders.eu' must translate to protectDefenders.eu
OK — 3 violations detected
OK — 0 violations on correct translation


### 1.3 Retry hint variants (A and B)

Two variants of the retry hint are kept side-by-side in `glossary.py`:

- Variant A (`retry_hint_minimal`) does not show the previous translation. Use when you want the model to re-translate cold with terms specified.
- Variant B (`retry_hint_with_previous`) shows the model its prior (incorrect) attempt, asks it to correct only the listed terms. Larger prompt; the model can edit instead of re-generate. This is the current default in `Translator.translate()`.

Swap between them by changing one line in [translate.py](translate.py) (`retry_hint_with_previous` vs `retry_hint_minimal`).

In [4]:
violations = g.violations(source, bad)

# Variant A: minimal hint — no previous translation reference
print("--- Variant A (retry_hint_minimal) ---")
hint_a = g.retry_hint_minimal(violations)
print(hint_a)
print()

# Variant B: includes the previous (failed) translation
print("--- Variant B (retry_hint_with_previous, the default) ---")
hint_b = g.retry_hint_with_previous(violations, bad)
print(hint_b)
print()

--- Variant A (retry_hint_minimal) ---
For this translation, use exactly these terms:
  - 'PDDH' must translate to Human Rights Defenders (HRDs) (or: HRDs)
  - 'Corte IDH' must translate to IACtHR (or: Inter-American Court)
  - 'protectDefenders.eu' must translate to protectDefenders.eu

--- Variant B (retry_hint_with_previous, the default) ---
You previously translated this passage as:
---
The women presented a report to the Court. See defenders.eu.
---
The previous translation had terminology errors. Correct only these specific terms; keep everything else unchanged:
  - 'PDDH' must translate to Human Rights Defenders (HRDs) (or: HRDs)
  - 'Corte IDH' must translate to IACtHR (or: Inter-American Court)
  - 'protectDefenders.eu' must translate to protectDefenders.eu



### 1.4 Parse Step 1a LLM output

Tests `DocumentReviewer._parse_keep_term_lines` against these scenarios:

1. Structured input - the format the prompt asks for (`KEEP:` / `TERM:` prefixes, `|` variant separator, `(Abbrev)` parenthetical splits, leading `-` bullets, `#` comments).
2. Bare-line fallback - translategemma's actual failure mode where it emits one term per line without any prefix. The parser recovers these via `_looks_like_term_candidate` (proper-noun-y, no sentence punctuation, reasonable length).

In [5]:
from entity_extract import DocumentReviewer

# Test 1: structured prefixes (the format the prompt asks for)
structured = """
KEEP: Leonardo Soto
TERM: Comisión Interamericana de Derechos Humanos | CIDH | Comisión
TERM: Corte Interamericana de Derechos Humanos (Corte IDH)
- KEEP: protectDefenders.eu
random line that should be skipped (lowercase prose with sentence punctuation.)
# a comment
"""

keep, groups = DocumentReviewer._parse_keep_term_lines(structured)
print(f"--- Structured input ---")
print(f"KEEP: {sorted(keep)}")
print(f"TERM groups: {groups}")

assert keep == {"Leonardo Soto", "protectDefenders.eu"}
assert ("Comisión Interamericana de Derechos Humanos", "CIDH", "Comisión") in groups
assert ("Corte Interamericana de Derechos Humanos", "Corte IDH") in groups
print()

# Test 2: bare-line fallback
# translategemma sometimes emits bare terms without the KEEP:/TERM: prefix. The parser should still recover them.
bare = """personas defensoras de derechos humanos (PDDH)
PDDH
Corte Interamericana de Derechos Humanos
Corte IDH
Comisión Interamericana de Derechos Humanos
CIDH
protectDefenders.eu"""

keep, groups = DocumentReviewer._parse_keep_term_lines(bare)
print(f"--- Bare-line input ---")
print(f"KEEP: {sorted(keep)}")
print(f"TERM groups:")
for g in groups:
    print(f"  {list(g)}")

--- Structured input ---
KEEP: ['Leonardo Soto', 'protectDefenders.eu']
TERM groups: [('Comisión Interamericana de Derechos Humanos', 'CIDH', 'Comisión'), ('Corte Interamericana de Derechos Humanos', 'Corte IDH')]

--- Bare-line input ---
KEEP: []
TERM groups:
  ['personas defensoras de derechos humanos', 'PDDH']
  ['Corte Interamericana de Derechos Humanos']
  ['Corte IDH']
  ['Comisión Interamericana de Derechos Humanos']
  ['CIDH']
  ['protectDefenders.eu']


## 2. Live Ollama sanity checks
Require `ollama serve` + `translategemma`.

### 2.1 Translator sanity check

In [6]:
from translate import Translator

t = Translator(source_lang="Spanish")
print(t.translate("Hola mundo, esto es una prueba."))
print(t.translate("personas defensoras de derechos humanos (PDDH)"))

Hello world, this is a test.
human rights defenders (HRDs)


### 2.2 DocumentReviewer on a small sample
Run the terminology-review pass on a short Spanish passage and inspect the glossary it produces.

In [8]:
from blocks import BodyPara, Run
from entity_extract import DocumentReviewer

sample = (
    "Históricamente Colombia ha registrado patrones persistentes de violencia y restricciones al espacio cívico contra personas defensoras de derechos humanos (PDDH), especialmente en escenarios socioambientales, de tierras y de disputa territorial."
    "El sistema risk_alert genera alertas tempranas. Visite protectDefenders.eu para más información. La CIDH también recibió el documento."
)
print(sample)
blocks = [BodyPara(runs=[Run(text=sample)])]

reviewer = DocumentReviewer(
    model="translategemma",
    source_lang="Spanish",
    target_lang="English",
)
glossary = reviewer.build_glossary(blocks, glossary_path="/tmp/test_glossary.txt")

print("\n--- Glossary file contents ---")
print(Path("/tmp/test_glossary.txt").read_text())

print("\n--- Parsed entries ---")
for e in glossary._entries:
    src = " | ".join(e.source_terms)
    alts = f"  (alts: {e.target_alts})" if e.target_alts else ""
    print(f"  [{e.kind}] {src} → {e.target}{alts}")

DomainGlossary.delete("/tmp/test_glossary.txt")

Históricamente Colombia ha registrado patrones persistentes de violencia y restricciones al espacio cívico contra personas defensoras de derechos humanos (PDDH), especialmente en escenarios socioambientales, de tierras y de disputa territorial.El sistema risk_alert genera alertas tempranas. Visite protectDefenders.eu para más información. La CIDH también recibió el documento.
  [entity_extract] identifying terms in 1 segment(s) ...
  [entity_extract] identification complete: 1 KEEP, 2 TERM group(s)
  [entity_extract] translating 2 canonical term(s) to English ...
  [entity_extract] demoted 1 identity translation(s) to KEEP
  [entity_extract] 3 total entries
  [entity_extract] glossary: 4 primary + 0 user-provided = 4 total entries
  [glossary] written to /tmp/test_glossary.txt

--- Glossary file contents ---
# Translation glossary — auto-generated from document review.
# You may edit this file before translation completes.
# Formats:  TRANSLATE: source → target   (enforced; triggers re

## 3. Debug / inspection tools

### 3.1 Debug pass 1 on a single segment

Drop any segment text into `segment` below to see exactly what step 1a produces. Use this to iterate `IDENTIFY_PROMPT` in `prompts.py` without re-running the whole document.

In [9]:
from entity_extract import DocumentReviewer

print(sample)

r = DocumentReviewer(model="translategemma",
                     source_lang="Spanish", target_lang="English")
raw, keep, translate_groups = r.identify_segment_debug(sample)

print("\n--- Raw LLM response ---")
print(raw)

print(f"\n--- Parsed ---")
print(f"KEEP: {sorted(keep)}")
print("TERM groups:")
for variants in translate_groups:
    print(f"  {list(variants)}")

Históricamente Colombia ha registrado patrones persistentes de violencia y restricciones al espacio cívico contra personas defensoras de derechos humanos (PDDH), especialmente en escenarios socioambientales, de tierras y de disputa territorial.El sistema risk_alert genera alertas tempranas. Visite protectDefenders.eu para más información. La CIDH también recibió el documento.

--- Raw LLM response ---
TERM: personas defensoras de derechos humanos | PDDH
TERM: protectDefenders.eu
KEEP: CIDH

--- Parsed ---
KEEP: ['CIDH']
TERM groups:
  ['personas defensoras de derechos humanos', 'PDDH']
  ['protectDefenders.eu']


### 3.2 Full snapshot dump

Pass `dump_dir=...` to capture every intermediate artifact in Phase 1: input segments, raw LLM responses, parsed results, the cross-segment merged state, and the final entries.

Files written per run:
- `segment_NNN_input.txt` — the segment text sent to step 1a
- `segment_NNN_step1a_prompt.txt` — the full prompt sent to the LLM
- `segment_NNN_step1a_raw.txt` — raw LLM response
- `segment_NNN_step1a_parsed.json` — parser output
- `step1b_input.json` — canonicals sent to step 1b
- `step1b_batch_NN_prompt.txt` / `_raw.txt` — per-batch prompts and responses
- `step1b_parsed.json` — step 1b translations
- `step1_merged.json` — cross-segment merged state before entry building
- `review_entries.json` — final GlossaryEntry list before write

In [10]:
import os, shutil
from blocks import BodyPara, Run
from entity_extract import DocumentReviewer

DUMP_DIR = "./glossary_debug_snapshot"
shutil.rmtree(DUMP_DIR, ignore_errors=True)

blocks = [BodyPara(runs=[Run(text=sample)])]
r = DocumentReviewer(model="translategemma",
                     source_lang="Spanish", target_lang="English",
                     dump_dir=DUMP_DIR)
entries = r.extract_terms(blocks)

print(f"\n--- Snapshot files in {DUMP_DIR} ---")
for f in sorted(os.listdir(DUMP_DIR)):
    size = os.path.getsize(os.path.join(DUMP_DIR, f))
    print(f"  {f}  ({size} bytes)")

  [entity_extract] identifying terms in 1 segment(s) ...
  [entity_extract] identification complete: 1 KEEP, 2 TERM group(s)
  [entity_extract] translating 2 canonical term(s) to English ...
  [entity_extract] demoted 1 identity translation(s) to KEEP
  [entity_extract] 3 total entries

--- Snapshot files in ./glossary_debug_snapshot ---
  review_entries.json  (503 bytes)
  segment_000_input.txt  (384 bytes)
  segment_000_step1a_parsed.json  (174 bytes)
  segment_000_step1a_prompt.txt  (2226 bytes)
  segment_000_step1a_raw.txt  (89 bytes)
  step1_merged.json  (236 bytes)
  step1b_batch_00_prompt.txt  (1231 bytes)
  step1b_batch_00_raw.txt  (114 bytes)
  step1b_input.json  (72 bytes)
  step1b_parsed.json  (121 bytes)


### 3.3 Document extraction
Extract a real document and look at the block structure (headings, body paragraphs, list items, tables, footnotes, comments).

In [14]:
from docx_extract import DocxExtractor
from blocks import Heading, BodyPara, ListItem, Footnote, Comment

extractor = DocxExtractor(DOCX_PATH)
blocks, tables = extractor.extract()
extractor.close()

by_type = {}
for b in blocks:
    by_type[type(b).__name__] = by_type.get(type(b).__name__, 0) + 1
print("Blocks by type:", by_type)
print(f"Tables: {len(tables)}")

print("\nFirst 5 text blocks:")
shown = 0
for b in blocks:
    if isinstance(b, (Heading, BodyPara)):
        print(f"  [{type(b).__name__}] {b.text[:150]}")
        shown += 1
        if shown >= 5:
            break

Blocks by type: {'BodyPara': 174, 'Heading': 9, 'TablePlaceholder': 1, 'Footnote': 88, 'Comment': 124}
Tables: 1

First 5 text blocks:
  [BodyPara] Blueprint
  [BodyPara] Lineamientos para el desarrollo de sistemas de información para la prevención de crímenes contra personas defensoras de derechos humanos
  [BodyPara] 15.06.2026
  [BodyPara] Revisada
  [Heading] 1. Introducción: sistemas de información para la prevención, protección y rendición de cuentas frente a la violencia contra personas defensoras


## 4. End-to-end translation

All subsections below use the same `DOC` and `OUT` set in the next cell. Pick a subsection based on what you want:
- 4.1 simplest: build glossary + translate in one call
- 4.2 two-phase: build glossary, inspect / edit it, then translate
- 4.3 debug: same as 4.1 but with `dump_dir` snapshots for prompt iteration
- 4.4 archival: timestamped output filenames + `translation_log.jsonl` run history

In [27]:
from translate import translate_document

# Swap between DOCX_PATH (full) and SHORT_DOCX_PATH.
DOC = SHORT_DOCX_PATH
OUT = f"./translated/{Path(DOC).stem}_translated.docx"
print(f"DOC = {DOC}\nOUT = {OUT}")

DOC = ./docs/short-docs/29.01. Blueprint Cejil comenta VRA_short2.docx
OUT = ./translated/29.01. Blueprint Cejil comenta VRA_short2_translated.docx


### 4.1 One-shot translation

Build the glossary and translate in one call.

In [28]:
result = translate_document(
    DOC, OUT,
    source_lang="Spanish", target_lang="English",
    verbose_glossary=True,   # log glossary entries injected per chunk
    keep_glossary=True,
    # force_rebuild=True,      # uncomment to discard the existing glossary
)
print(f"\nChars in:  {result.get('chars_in')}")
print(f"Output:    {result.get('output', OUT)}")
print(f"Glossary:  {result.get('glossary_path')}")

  [entity_extract] identifying terms in 2 segment(s) ...
  [entity_extract] reclassified 3 KEEP → TERM (ordinary phrases): ['entidades', 'estado de cosas inconstitucional', 'personas defensoras de derechos humanos']
  [entity_extract] absorbed 12 KEEP(s) into existing TRANSLATE groups: ['Corte', 'Corte Constitucional', 'Corte Constitucional de Colombia', 'Corte IDH', 'Corte Interamericana de Derechos Humanos', 'La Corte', 'La Corte Constitucional', 'La Corte Constitucional de Colombia', 'La Corte Interamericana de Derechos Humanos', 'La Sentencia SU-546 de 2023', 'La sentencia Cajar vs. Colombia', 'PDDH']
  [entity_extract] consolidated 14 groups into 13 by shared variants
  [entity_extract] identification complete: 0 KEEP, 13 TERM group(s)
  [entity_extract] translating 13 canonical term(s) to English ...
  [entity_extract] 2 term(s) untranslated by step 1b — marked as PREFER (soft guidance, no enforcement)
  [entity_extract] 13 total entries
  [entity_extract] glossary: 14 primary + 

### 4.2 Two-phase workflow (build, edit, translate)

1. Run Phase 1 to build the glossary at `{OUT}_glossary.txt`. `force_rebuild=True` discards any prior glossary at the expected path.
2. Open the file, review / edit it (fix wrong translations, change TRANSLATE to KEEP, delete junk, merge variants with `|`).
3. Run Phase 2 to translate using the (possibly-edited) glossary.

In [29]:
result = translate_document(
    DOC, OUT,
    source_lang="Spanish", target_lang="English",
    phases=("build_glossary",),
    force_rebuild=True,
)
glossary_path = result["glossary_path"]
print(f"\nGlossary written to {glossary_path}")
print("Open it, review / edit, then run the Phase 2 cell below.")
print("\n--- First 30 lines of the glossary ---")
print("\n".join(Path(glossary_path).read_text().splitlines()[:30]))

  [translate_document] force_rebuild=True — removing existing glossary at translated/29.01. Blueprint Cejil comenta VRA_short2_translated_glossary.txt
  [entity_extract] identifying terms in 2 segment(s) ...
  [entity_extract] reclassified 3 KEEP → TERM (ordinary phrases): ['entidades', 'estado de cosas inconstitucional', 'personas defensoras de derechos humanos']
  [entity_extract] absorbed 12 KEEP(s) into existing TRANSLATE groups: ['Corte', 'Corte Constitucional', 'Corte Constitucional de Colombia', 'Corte IDH', 'Corte Interamericana de Derechos Humanos', 'La Corte', 'La Corte Constitucional', 'La Corte Constitucional de Colombia', 'La Corte Interamericana de Derechos Humanos', 'La Sentencia SU-546 de 2023', 'La sentencia Cajar vs. Colombia', 'PDDH']
  [entity_extract] consolidated 14 groups into 13 by shared variants
  [entity_extract] identification complete: 0 KEEP, 13 TERM group(s)
  [entity_extract] translating 13 canonical term(s) to English ...
  [entity_extract] 2 term(s) un

In [30]:
result = translate_document(
    DOC, OUT,
    source_lang="Spanish", target_lang="English",
    phases=("translate",),
    verbose_glossary=True,
    keep_glossary=True,
)
print(f"\nChars in:  {result.get('chars_in')}")
print(f"Output:    {result.get('output', OUT)}")

Extracted 37 blocks (10207 chars, 2 table(s), 0 image(s), 12 footnote(s), 6 comment(s))
  [glossary] chunk 'El informe de la CIDH sobre situación de personas defensoras de derechos humanos':
    Required terminology — use these translations exactly whenever the source term appears:
      CIDH → Inter-American Human Rights Court
      personas defensoras de derechos humanos → human rights defenders
  Glossary violation(s) — retrying with correction hint:
    • 'CIDH' must translate to Inter-American Human Rights Court
  [glossary] chunk 'El informe de la CIDH sobre situación de personas defensoras de derechos humanos':
    Required terminology — use these translations exactly whenever the source term appears:
      CIDH → Inter-American Human Rights Court
      personas defensoras de derechos humanos → human rights defenders
  Retry did not improve violations; keeping first result.
  Translating 4 paragraph(s) with footnotes...
  [glossary] chunk 'Recientemente, la CIDH publicó el Terce

### 4.3 With snapshots

Same as 4.1 but passes `dump_dir` to capture every Phase 1 intermediate artifact (per-segment input / prompt / raw response / parsed groups, plus the merged state and final entries). Useful for iterating on `IDENTIFY_PROMPT` / `TRANSLATE_TERMS_PROMPT` or diagnosing weird glossary entries.

In [32]:
DUMP_DIR = str(Path(OUT).parent / f"{Path(OUT).stem}_snapshots")
print(f"DUMP_DIR: {DUMP_DIR}")
shutil.rmtree(DUMP_DIR, ignore_errors=True)

result = translate_document(
    DOC, OUT,
    source_lang="Spanish", target_lang="English",
    verbose_glossary=True,
    keep_glossary=True,
    dump_dir=DUMP_DIR,
    force_rebuild=True,
)
print(f"\nGlossary:  {result.get('glossary_path')}")
print(f"Snapshots in {DUMP_DIR}/")

DUMP_DIR: translated/29.01. Blueprint Cejil comenta VRA_short2_translated_snapshots
  [entity_extract] identifying terms in 2 segment(s) ...
  [entity_extract] reclassified 3 KEEP → TERM (ordinary phrases): ['entidades', 'estado de cosas inconstitucional', 'personas defensoras de derechos humanos']
  [entity_extract] absorbed 12 KEEP(s) into existing TRANSLATE groups: ['Corte', 'Corte Constitucional', 'Corte Constitucional de Colombia', 'Corte IDH', 'Corte Interamericana de Derechos Humanos', 'La Corte', 'La Corte Constitucional', 'La Corte Constitucional de Colombia', 'La Corte Interamericana de Derechos Humanos', 'La Sentencia SU-546 de 2023', 'La sentencia Cajar vs. Colombia', 'PDDH']
  [entity_extract] consolidated 14 groups into 13 by shared variants
  [entity_extract] identification complete: 0 KEEP, 13 TERM group(s)
  [entity_extract] translating 13 canonical term(s) to English ...
  [entity_extract] 2 term(s) untranslated by step 1b — marked as PREFER (soft guidance, no enforce

### 4.4 With timestamped outputs + run log

`timestamp=True` inserts the current time into the output stem so each run keeps its own docx + glossary pair (`outline_blueprint_2026-07-02_1435.docx` + `..._glossary.txt`). Every call to `translate_document` also appends one JSON line to `<output_dir>/translation_log.jsonl` with timestamp, input, output, glossary path, phases, and model.

In [ ]:
result = translate_document(
    DOC, OUT,
    source_lang="Spanish", target_lang="English",
    verbose_glossary=True,
    keep_glossary=True,
    timestamp=True,
)
print(f"\nChars in:  {result.get('chars_in')}")
print(f"Output:    {result.get('output', OUT)}")
print(f"Glossary:  {result.get('glossary_path')}")

## 5. Batch translate a directory

In [ ]:
from batch_translate import batch_translate

BATCHTRANSLATE_DIRPATH_INPUT = "./docs/short-docs"
BATCHTRANSLATE_DIRPATH_OUTPUT = BATCHTRANSLATE_DIRPATH_INPUT + "__batch_output"
results = batch_translate(BATCHTRANSLATE_DIRPATH_INPUT, BATCHTRANSLATE_DIRPATH_OUTPUT)